# NUS prerequisite graph construction

This notebook constructs prerequisite-based artefacts used in downstream curriculum analysis.

## Objective
The workflow builds a directed prerequisite view for NUS undergraduate modules by:
1. fetching the NUS module catalogue for AY `2025-2026`
2. filtering to graded undergraduate modules with usable metadata
3. querying NUSMods for prerequisite trees
4. converting module-level prerequisites into a reverse dependency view
5. handling wildcard prerequisite tokens
6. deduplicating downstream modules by title/description identity

### Generated outputs
- `direct_prereqs.json` — mapping from each module to its direct prerequisites
- `dependent_mods_dedup.csv` — cleaned and deduplicated mapping from prerequisite modules to the modules they unlock

## Notes
- A cached prerequisite lookup is used to reduce repeated API calls.
- The notebook preserves the original logic and intermediate reload of `dependent_mods_info.csv`.


## 1. Imports

In [ ]:
import json
import os
import pickle
import re
import time
from collections import defaultdict
from typing import Any, Dict, List, Optional, Set

import pandas as pd
import requests

## 2. Load and filter the NUS module catalogue

In [ ]:
# Fetch the full NUS module catalogue for the latest academic year.
# The raw JSON is filtered in the next cell to keep only the undergraduate modules used in the project.
MODULE_LIST_URL = "https://api.nusmods.com/v2/2025-2026/moduleInfo.json"

response = requests.get(MODULE_LIST_URL, timeout=30)
response.raise_for_status()
data = response.json()

In [ ]:
# Filter the module list based on the scope of the project:
# - graded modules only
# - undergraduate modules only (level 4k and below)
# - non-empty description, faculty, and department (for downstream EDA if necessary)

def is_undergrad(code: str) -> bool:
    match = re.search(r"\d+", str(code))
    return bool(match) and not match.group().startswith("5")

filtered_data = []

for d in data:
    # 1. Check grading basis and undergrad status
    if d.get("gradingBasisDescription") == "Graded" and is_undergrad(d.get("moduleCode")):
        # 2. Extract values safely
        desc = str(d.get("description", "")).strip()
        fac = str(d.get("faculty", "")).strip()
        dept = str(d.get("department", "")).strip()

        # 3. Ensure required fields are not empty/whitespace
        if desc and fac and dept:
            filtered_data.append({
                "moduleCode": d.get("moduleCode"),
                "title": d.get("title"),
                "description": desc,
                "faculty": fac,
                "department": dept
            })

module_base_df = pd.DataFrame(filtered_data)
module_base_df.rename(columns={"moduleCode": "module_code"}, inplace=True)
module_base_df["module_code"] = module_base_df["module_code"].astype(str).str.strip()
module_base_df = module_base_df.drop_duplicates(subset=["module_code"]).reset_index(drop=True)

valid_module_codes: Set[str] = set(module_base_df["module_code"])

## 3. Configure API session and cache

In [ ]:
# Use a persistent requests session and a local cache to avoid repeated calls on NUSMods API for prerequisite trees that were already fetched.
session = requests.Session()
session.headers.update({"Accept": "application/json"})

def save_cache(cache: Dict[str, Any], path: str = "data/prereq_cache.pkl") -> None:
    os.makedirs("data", exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(cache, f)

def load_cache(path: str = "data/prereq_cache.pkl") -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return {}

prereq_cache = load_cache()
print(f"Loaded cache with {len(prereq_cache)} modules")

## 4. Fetch prerequisite trees using helper functions

In [ ]:
# Fetch a module's prerequisite tree from NUSMods.
# If the current academic year does not expose the module cleanly, the function falls back across earlier academic years down to 2020-2021.
def fetch_prereq_info(module_code: str, acad_year: str = "2025-2026") -> Optional[Dict[str, Any]]:
    """
    Fetch prereqTree for a module, trying the given academic year first,
    then earlier years down to 2020-2021.

    Returns:
        prereqTree if found, else None
    """
    latest_year = int(acad_year.split("-")[0])

    while latest_year >= 2020:
        curr_acad_year = f"{latest_year}-{latest_year + 1}"
        url = f"https://api.nusmods.com/v2/{curr_acad_year}/modules/{module_code}.json"

        try:
            response = session.get(url, timeout=60)
            if response.status_code == 404:
                latest_year -= 1
                continue

            response.raise_for_status()
            module_json = response.json()
            return module_json.get("prereqTree")

        except requests.exceptions.RequestException:
            latest_year -= 1
            continue

    return None

def normalize_module_code(code: str) -> str:
    """Remove suffixes like :D from module codes."""
    if ":" in code:
        return code.split(":")[0]
    return code

def extract_all_prereqs(prereq_tree: Dict) -> Set[str]:
    """Recursively extract all module codes from prerequisite tree."""
    if not prereq_tree:
        return set()

    prereqs = set()

    if isinstance(prereq_tree, str):
        prereqs.add(normalize_module_code(prereq_tree))

    elif isinstance(prereq_tree, dict):
        for value in prereq_tree.values():
            prereqs.update(extract_all_prereqs(value))

    elif isinstance(prereq_tree, list):
        for item in prereq_tree:
            prereqs.update(extract_all_prereqs(item))

    return prereqs

def build_direct_prereqs(module_codes: List[str], cache: Dict[str, Any], save_every: int = 500) -> Dict[str, List[str]]:
    result = {}

    for i, module_code in enumerate(module_codes, start=1):
        print(f"[{i}/{len(module_codes)}] {module_code}")

        if module_code in cache:
            prereq_tree = cache[module_code]
        else:
            try:
                prereq_tree = fetch_prereq_info(module_code)
            except Exception as e:
                print(f"Failed for {module_code}: {e}")
                prereq_tree = None

            cache[module_code] = prereq_tree

        if not prereq_tree:
            result[module_code] = []
        else:
            result[module_code] = sorted(extract_all_prereqs(prereq_tree))

        if i % save_every == 0:
            save_cache(cache)
            print(f"Checkpoint saved at {i} modules")
            time.sleep(0.1)

    save_cache(cache)
    return result

In [ ]:
# Extract the filtered module list that will be queried for prerequisite information.
module_codes = module_base_df["module_code"].tolist()
len(module_codes)


`build_direct_prereqs(...)` queries or reuses cached prerequisite trees for every filtered module code. The output is a dictionary of the form:

- **key** = module code
- **value** = list of direct prerequisite tokens needed for that module


In [ ]:
# Build the direct prerequisite dictionary:

direct_prereqs = build_direct_prereqs(
    module_codes,
    cache=prereq_cache
)

## 5. Save and reload direct prerequisites

In [ ]:
# Same output fir future use to avoid re-querying prerequisite trees from the API.
with open("direct_prereqs.json", "w") as file:
    json.dump(direct_prereqs, file, indent=2)

print("Saved direct_prereqs JSON to direct_prereqs.json")

In [ ]:
# Read back in after initial run if needed
with open("direct_prereqs.json", "r") as file:
    direct_prereqs = json.load(file)

## 6. Build reverse dependency map

In [ ]:
# Reverse the mapping so each prerequisite token points to the modules that directly depend on it
reverse_map = defaultdict(set)

for module_code, prereqs in direct_prereqs.items():
    if not prereqs:
        continue

    for prereq in set(prereqs):   
        reverse_map[prereq].add(module_code)

### Reverse mapping logic
Downstream analyses uses **prerequisite → dependent modules** information so that a module's structural importance can be measured by how many later modules it unlocks

### Wildcard prerequisite tokens
Wildcard modules are those with `%` behind to specify different courses.  
These are inspected separately because some wildcard patterns are overly broad and would inflate downstream counts if treated as exact prerequisite modules.

In [ ]:
# Convert the reverse map into a token-level table for inspection and filtering.
# Each row represents one prerequisite token and the list of modules it unlocks.
reverse_count_df = pd.DataFrame([
    {
        "prereq_token": prereq,
        "dependent_modules": sorted(dependents)
    }
    for prereq, dependents in reverse_map.items()
])

reverse_count_df["is_wildcard"] = reverse_count_df["prereq_token"].str.contains("%", regex=False)
reverse_count_df.head(20)

**Wildcard categories**

- **Broad wildcards**: Module codes with fewer than four digits (e.g. `ST36%`), which are too general to uniquely identify a specific module. These may correspond to a wide range of modules across different levels or programmes.

- **Narrow wildcards**: Module codes that are more specific (e.g. `CS1010%`), typically referring to a small set of closely related module variants that share a common base code.

In [ ]:
# Classify wildcard prerequisite tokens into broad or narrow patterns.

def classify_wildcard(row):
    token = row["prereq_token"]

    # Only classify wildcard rows
    if not row["is_wildcard"]:
        return None

    prefix = token.replace("%", "")

    # Extract digits
    digits = re.findall(r"\d", prefix)
    n_digits = len(digits)

    if n_digits == 4:
        return "narrow"
    else:
        return "broad"

reverse_count_df["wildcard_type"] = reverse_count_df.apply(classify_wildcard, axis=1)
reverse_count_df.head()

In [ ]:
# Keep:
# - all non-wildcard prerequisite tokens
# - only narrow wildcard tokens (4 digits in module code so unique module can be identified)

focused_df = reverse_count_df[
    (reverse_count_df["is_wildcard"] == False) |
    (reverse_count_df["wildcard_type"] == "narrow")
].copy()

focused_df.head(20)

### Why deduplicate dependent modules?
During preprocessing, it was observed that module variants (often denoted using wildcard symbols such as %) do not always map cleanly to a single underlying module. In some cases, multiple variants correspond to the same content, while in others, they represent distinct modules designed for different degree programmes.

Without deduplication, this would lead to inflated counts of downstream modules.

**Examples**

- `MKT1705%`  
  → `MKT1705A`, `MKT1705B`, `MKT1705C`  
  → All correspond to the **same module content** within business programmes.

- `CS1010S%`  
  → `CS1010S`, `CS1010A`, `CS1010J`  
  → Correspond to **different module variants** designed for different degree programmes, with different learning content.

To address this, we construct a **title- and description-based** identity key so that downstream analyses avoid incorrectly merging distinct modules or overcounting equivalent ones.

## 7. Prepare dependent module information

In [ ]:
# Collect the full set of downstream modules that appear after wildcard filtering.
# These are then matched back to module metadata for identity-based deduplication.
all_dependent_modules = sorted({
    mod
    for mods in focused_df["dependent_modules"]
    if isinstance(mods, list)
    for mod in mods
})

In [ ]:
# Pull module metadata for all downstream modules currently referenced.
# This intermediate dataframe is subsequently replaced by the original CSV reload

dependent_modules_df = module_base_df[module_base_df["module_code"].isin(all_dependent_modules)].copy()

The original workflow then reloads `dependent_mods_info.csv` before identity-based deduplication.  
This is preserved here to avoid changing the project logic.

In [ ]:
# Preserve the original workflow by reloading the prepared dependent module information from CSV before constructing identity keys.
dependent_modules_df = pd.read_csv("dependent_mods_info.csv")

In [ ]:
# Construct a simple identity key from normalized title and description so that odules with different codes but effectively identical content can be deduplicated.
def normalize_text_for_identity(text):
    if text is None:
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

dependent_modules_df["title_norm"] = dependent_modules_df["title"].apply(normalize_text_for_identity)
dependent_modules_df["description_norm"] = dependent_modules_df["description"].apply(normalize_text_for_identity)

dependent_modules_df["identity"] = (
    dependent_modules_df["title_norm"] + " || " + dependent_modules_df["description_norm"]
)

# Fallback if both missing
dependent_modules_df.loc[
    dependent_modules_df["identity"] == " || ", "identity"
] = "CODE_ONLY::" + dependent_modules_df["module_code"]

In [ ]:
# Build lookup tables:
# - module_to_identity: module code -> normalized content identity
# - identity_to_rep:    normalized identity -> representative module code
module_to_identity = dict(
    zip(dependent_modules_df["module_code"], dependent_modules_df["identity"])
)

identity_to_rep = (
    dependent_modules_df.groupby("identity")["module_code"]
    .min()
    .to_dict()
)

In [ ]:
# Replace duplicate module variants within each dependent-module list using the normalized identity key, keeping one representative code per identity.
def deduplicate_dependent_modules(mod_list, module_to_identity, identity_to_rep):
    if not isinstance(mod_list, list):
        return []

    seen_identities = set()
    deduped = []

    for mod in mod_list:
        identity = module_to_identity.get(mod, f"CODE_ONLY::{mod}")
        if identity not in seen_identities:
            seen_identities.add(identity)
            deduped.append(identity_to_rep.get(identity, mod))

    return deduped

In [ ]:
# Apply identity-based deduplication to each list of dependent modules.
focused_df["dependent_modules_dedup"] = focused_df["dependent_modules"].apply(
    lambda mods: deduplicate_dependent_modules(mods, module_to_identity, identity_to_rep)
)

### Exported counts
This section computes both the raw and deduplicated number of downstream modules unlocked by each prerequisite token. 

## 8. Compute counts and export

In [ ]:
# Compare raw vs deduplicated downstream counts for each prerequisite token.
# `inflation_removed` shows how much double-counting was eliminated.
focused_df["n_direct_dependents_raw"] = focused_df["dependent_modules"].apply(
    lambda mods: len(mods) if isinstance(mods, list) else 0
)

focused_df["n_direct_dependents_dedup"] = focused_df["dependent_modules_dedup"].apply(len)

focused_df["inflation_removed"] = (
    focused_df["n_direct_dependents_raw"] - focused_df["n_direct_dependents_dedup"]
)

focused_df.sort_values(by="n_direct_dependents_dedup", ascending=False, inplace=True)
focused_df.head()

In [ ]:
# Final export used downstream.
# This CSV stores the filtered prerequisite-token table together with deduplicated dependent-module lists and downstream count summaries.
focused_df.to_csv("dependent_mods_dedup.csv", index=False)
print("Saved dependent_mods_dedup.csv")